# Aktywacja Strategii Odbicie Ogórkowe
Ten notatnik służy do testowania, wizualizacji i optymalizacji strategii powrotu do średniej po mocnych spadkach.

## Importy, dane i sygnały

In [1]:
import os
import sys

# Dodajemy folder glowny do path aby moduly dzialaly
sys.path.append(os.path.abspath('d:/Antigravity/rebound/lyse-lby'))
os.chdir('d:/Antigravity/rebound/lyse-lby')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Importujemy ladowanie danych i nasze nowe moduly
from maciex_py.ladowanie_danych import create_stock_dfs
from odbicie_dlugie.mackowe_sygnaly import mackowe_sygnaly
from odbicie_dlugie.odbicie import generate_odbicie_entries
from odbicie_dlugie.tbm import moving_triple_barrier_labels

# Ustawienia ładowania danych
settings = {
    'market': 'stocks',
    'interval': '1week',
    'vol_enabled': True,
    'vol_ratio_window': 20,
    'vol_ratio_threshold': 1.2,
    'cmo_enabled': True,
    'cmo_len': 6,
    'cmo_thres': -35,
    'cmo_thres_prev': -50
}

In [2]:
# 1. Ładowanie Danych
import pickle
import os

data_cache_file = 'dfs_cache.pkl'

if os.path.exists(data_cache_file):
    print("Znaleziono zapisane dane. Wczytywanie z pliku...")
    with open(data_cache_file, 'rb') as f:
        dfs_1d, dfs_1w = pickle.load(f)
    print(f"Wczytano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W z pliku {data_cache_file}.")
else:
    print("Ładowanie danych dziennych i tygodniowych...")
    dfs_1d, dfs_1w = create_stock_dfs(settings)
    print(f"Załadowano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W.")
    print("Zapisywanie danych do pliku...")
    with open(data_cache_file, 'wb') as f:
        pickle.dump((dfs_1d, dfs_1w), f)
    print("Dane zapisane pomyślnie.")


Znaleziono zapisane dane. Wczytywanie z pliku...
Wczytano 478 symboli 1D i 478 symboli 1W z pliku dfs_cache.pkl.


In [3]:
# 2. Generowanie Sygnałów Bazowych (mackowe_sygnaly)
signals_cache_file = 'signals_cache.pkl'

if os.path.exists(signals_cache_file):
    print("Znaleziono zapisane sygnały. Wczytywanie z pliku...")
    with open(signals_cache_file, 'rb') as f:
        signals_df = pickle.load(f)
    print(f"Wczytano {len(signals_df)} sygnałów z pliku {signals_cache_file}.")
else:
    signals_df = mackowe_sygnaly(
        dfs=dfs_1w,
        settings=settings,
        require_vol_confirmation=True,
        require_cmo_confirmation=True,
        interval='1w',
        entry_offset=0,
        pattern_cols=['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line'],
        debug=True
    )
    print("Zapisywanie sygnałów do pliku...")
    with open(signals_cache_file, 'wb') as f:
        pickle.dump(signals_df, f)
    print("Sygnały zapisane pomyślnie.")

signals_df.describe()


Znaleziono zapisane sygnały. Wczytywanie z pliku...
Wczytano 912 sygnałów z pliku signals_cache.pkl.


,signal_time,entry_time,signal_close
count,912,912,912.000000
mean,2023-10-08 09:01:34.736842,2023-10-08 09:01:34.736842,129.103488
min,2021-07-25 00:00:00,2021-07-25 00:00:00,7.970000
25%,2022-05-15 00:00:00,2022-05-15 00:00:00,51.740002
50%,2023-10-08 00:00:00,2023-10-08 00:00:00,97.340000
75%,2025-03-16 00:00:00,2025-03-16 00:00:00,179.127495
max,2026-02-22 00:00:00,2026-02-22 00:00:00,565.369995
std,NaN,NaN,103.256636


## Wejście i Wyjście

In [4]:
# 3. Wejście na podstawie progu (Odbicie Ogórkowe)
threshold_pct = 0.11  # 3% spadek od zamknięcia świecy sygnałowej

entries_df = generate_odbicie_entries(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    threshold_pct=threshold_pct,
    max_setup_hold_bars=10
)
print(f"Wygenerowano {len(entries_df)} wejść przy progu {threshold_pct*100}%")
entries_df.head()

Wygenerowano 107 wejść przy progu 11.0%


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr
0,ABNB,2022-06-19,hammer,2022-06-30,88.546098,99.489998,0.11,6.897889
1,ALB,2023-11-05,hammer,2023-11-09,113.902203,127.980003,0.11,7.207769
2,ALB,2023-10-29,inverted_hammer,2023-11-01,119.651602,134.440002,0.11,7.000488
3,ARE,2025-11-23,hammer,2025-12-08,45.292099,50.889999,0.11,2.208598
4,AXP,2025-03-23,engulfing_bull,2025-04-04,237.919998,270.510010,0.11,10.113220


In [5]:
# Params           Więcej:                                                              Mniej:
tpm = 1.25    #    - łapiemy większe ruchy (ryzykujemy powrotem).                        - ratujemy i szybciej zbieramy mniejsze kwoty.
slm = 2.5     #    - luźniejszy stop loss (wytrzymuje szum korekcyjny).                  - szybsza kapitulacja i ucinanie straty z palca.
ttpm = 0.25   #    - luźniejsze spuszczanie kursu w trendach, nie zostajemy wyrzuceni.   - szybsze zabezpieczanie małego peaku.
mhb = 15      #    - dajemy kapitałowi długo leżeć pod ruchem bocznym.                   - szukamy szybkich obrotów uwalnaijąc portfel.

# Nowe parametry wyjścia czasowego i ochrony kapitału
early_bailout = False     # Ucieczka w połowie czasu (mhb/2) jeśli trade jest na minusie
time_decay_sl = False     # Stop loss podnosi się z czasem w kierunku ceny wejścia
active_trail_sl = False   # Stop loss podąża za każdym nowym szczytem, nie tylko po aktywacji TP
sl_trail_mult = 3.0       # Z jakiej odległości (w ATR) ma podążać aktywny SL
max_loss_pct = 0.15       # Twardy cap straty (15%). Chroni przed ogromnymi ATR-ami na groszówkach.

In [38]:
# Params V => overfitted, ale z głową
tpm = 1.2
slm = 2.7
ttpm = 0.1
mhb = 15               # disabled
early_bailout = False   # disabled
time_decay_sl = True
active_trail_sl = True 
sl_trail_mult = 3.0
max_loss_pct = 1        # disabled
exit_on_close=True


In [34]:
# 4. Wyjście z użyciem Moving Triple Barrier Method
trades_df = moving_triple_barrier_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=tpm,
    sl_mult=slm,
    tp_trail_mult=ttpm,
    max_holding_bars=mhb,
    early_breakeven=early_bailout,
    time_decay_sl=time_decay_sl,
    active_trailing_sl=active_trail_sl,
    sl_trail_mult=sl_trail_mult,
    max_loss_pct=max_loss_pct,
    exit_on_close=exit_on_close
)
print(f"Zakończono {len(trades_df)} transakcji.")
trades_df.head()


Zakończono 107 transakcji.


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr,exit_time,exit_price,return_pct,exit_reason,hold_bars
0,ABNB,2022-06-19,hammer,2022-06-30,88.546098,99.489998,0.11,6.897889,2022-07-11,95.099998,7.401682,TRAILING_TP,6
1,ALB,2023-11-05,hammer,2023-11-09,113.902203,127.980003,0.11,7.207769,2023-11-16,122.599998,7.636196,TRAILING_TP,5
2,ALB,2023-10-29,inverted_hammer,2023-11-01,119.651602,134.440002,0.11,7.000488,2023-11-06,119.459999,-0.160134,TRAILING_TP,3
3,ARE,2025-11-23,hammer,2025-12-08,45.292099,50.889999,0.11,2.208598,2025-12-19,47.939999,5.846272,TRAILING_TP,9
4,AXP,2025-03-23,engulfing_bull,2025-04-04,237.919998,270.510010,0.11,10.113220,2025-04-10,246.889999,3.770175,TRAILING_TP,4


## Analiza

In [35]:
# 5. Analiza i Statystyki
if not trades_df.empty:
    wins = (trades_df['return_pct'] > 0).sum()
    losses = (trades_df['return_pct'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    trades_df['return_per_bar'] = trades_df['return_pct'] / trades_df['hold_bars']
    
    print(f"Total Trades: {len(trades_df)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg Return: {trades_df['return_pct'].mean():.2f}%")
    print(f"Avg bars held: {trades_df['hold_bars'].mean():.2f}")
    print(f"Avg Return per Bar: {trades_df['return_per_bar'].mean():.2f}%")

    
    # Powody wyjścia
    print("\nExit Reasons:")
    print(trades_df['exit_reason'].value_counts())
else:
    print("Brak transakcji do analizy.")

Total Trades: 107
Win Rate: 80.37%
Avg Return: 6.11%
Avg bars held: 11.02
Avg Return per Bar: 1.04%

Exit Reasons:
exit_reason
TRAILING_TP    92
TRAILING_SL    14
SL              1
Name: count, dtype: int64


In [36]:
temp = pd.DataFrame({
    'count': trades_df.groupby('exit_reason').return_pct.count(),
    'avg_return': trades_df.groupby('exit_reason').return_pct.mean(),
    'cumulativ_return': trades_df.groupby('exit_reason').return_pct.sum(),
    'std': trades_df.groupby('exit_reason').return_pct.std(),
    'avg_hold_bars': trades_df.groupby('exit_reason').hold_bars.mean(),
    'std_hold_bars': trades_df.groupby('exit_reason').hold_bars.std(),
    'max_hold_bars': trades_df.groupby('exit_reason').hold_bars.max()
    })
    
temp

,count,avg_return,cumulativ_return,std,avg_hold_bars,std_hold_bars,max_hold_bars
exit_reason,,,,,,,
SL,1,-13.406540,-13.406540,NaN,1.000000,NaN,1
TRAILING_SL,14,-15.482631,-216.756829,6.359662,22.000000,12.347532,44
TRAILING_TP,92,9.609483,884.072395,9.095764,9.456522,10.541819,54


## Ploty

In [37]:
module_path = r"d:\Antigravity\rebound\lyse-lby\odbicie"
if module_path not in sys.path:
    sys.path.append(module_path)
    
from odbicie_dlugie.candlestick_cell import show_trade_viewer
show_trade_viewer(trades_df, dfs_1d, tpm, slm, ttpm, mhb,
                exit_reason='TRAILING_TP',
                active_trailing_sl=active_trail_sl,
                sl_trail_mult=sl_trail_mult,
                max_loss_pct=max_loss_pct,
                time_decay_sl=time_decay_sl,
                exit_on_close=exit_on_close)

Output()

## Porównanie

In [11]:
from odbicie_dlugie.tbm import optimize_simple_sl_tp
tp_mults = [a/100 for a in range(100, 300, 20)]  # Przetestuj TP od 1 do 3
sl_mults = [a/100 for a in range(100, 300, 20)]  # Przetestuj SL od 1 do 3
max_holding_bars = [15]
print("Uruchamianie optymalizacji prostej strategii SL/TP...")
simple_opt_results = optimize_simple_sl_tp(
    entries_df=entries_df, 
    market_data_daily=dfs_1d, 
    tp_mults=tp_mults, 
    sl_mults=sl_mults, 
    max_holding_bars_list=max_holding_bars,
    exit_on_close=True
)
display(simple_opt_results.head(10))

Uruchamianie optymalizacji prostej strategii SL/TP...


Optymalizacja Simple SL/TP:   0%|          | 0/100 [00:00<?, ?it/s]

,tp_mult,sl_mult,max_holding_bars,trades,win_rate,avg_return,avg_hold_bars,return_per_bar
98,2.8,2.6,15,107,69.158879,6.096661,13.065421,0.466626
99,2.8,2.8,15,107,69.158879,6.077410,13.074766,0.464820
68,2.2,2.6,15,107,71.028037,6.061791,11.598131,0.522652
69,2.2,2.8,15,107,71.028037,6.042541,11.607477,0.520573
96,2.8,2.2,15,107,68.224299,5.982127,13.000000,0.460164
66,2.2,2.2,15,107,70.093458,5.947258,11.532710,0.515686
78,2.4,2.6,15,107,69.158879,5.945011,12.299065,0.483371
79,2.4,2.8,15,107,69.158879,5.925760,12.308411,0.481440
97,2.8,2.4,15,107,68.224299,5.907986,13.018692,0.453808
58,2.0,2.6,15,107,72.897196,5.888297,11.149533,0.528120


In [17]:
from odbicie_dlugie.tbm import moving_triple_barrier_labels, simple_sl_tp_labels

simple_trades_df = simple_sl_tp_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=2.8,          # Sztywny TP
    sl_mult=2.6,          # Sztywny SL
    max_holding_bars=mhb, # Zamknięcie na czas
    exit_on_close=False    # Wyjście tylko po cenie Close
)

print(f"Zakończono {len(simple_trades_df)} prostych transakcji SL/TP.")
# i skopiować kod ze statystykami żeby zobaczyć różnicę w winrate i średnim zwrocie


Zakończono 107 prostych transakcji SL/TP.


In [18]:
import pandas as pd

# Upewnij się, że obie DataFrame nie są puste przed analizą
if not trades_df.empty and not simple_trades_df.empty:
    
    # --- Funkcja pomocnicza do obliczania statystyk ---
    def calc_stats(df, name):
        wins = (df['return_pct'] > 0).sum()
        total = len(df)
        win_rate = (wins / total * 100) if total > 0 else 0
        avg_ret = df['return_pct'].mean()
        avg_bars = df['hold_bars'].mean()
        ret_per_bar = avg_ret / avg_bars if avg_bars > 0 else 0
        
        return {
            'Strategy': name,
            'Total Trades': total,
            'Win Rate (%)': round(win_rate, 2),
            'Avg Return (%)': round(avg_ret, 2),
            'Avg Hold Bars': round(avg_bars, 2),
            'Return / Bar (%)': round(ret_per_bar, 4)
        }

    # --- Obliczanie i wyświetlanie ---
    tbm_stats = calc_stats(trades_df, "TBM (Trailing)")
    simple_stats = calc_stats(simple_trades_df, "Simple SL/TP")
    
    comparison_df = pd.DataFrame([tbm_stats, simple_stats])
    
    print("=== PORÓWNANIE STRATEGII ===")
    display(comparison_df.set_index('Strategy'))
    
    # --- Porównanie powodów wyjścia ---
    print("\nPowody wyjścia (TBM):")
    display(trades_df['exit_reason'].value_counts().to_frame('Liczba'))
    
    print("\nPowody wyjścia (Simple SL/TP):")
    display(simple_trades_df['exit_reason'].value_counts().to_frame('Liczba'))

else:
    print("Brak danych w trades_df lub simple_trades_df do porównania.")


=== PORÓWNANIE STRATEGII ===


,Total Trades,Win Rate (%),Avg Return (%),Avg Hold Bars,Return / Bar (%)
Strategy,,,,,
TBM (Trailing),107,80.37,6.11,11.02,0.5546
Simple SL/TP,107,69.16,7.68,23.36,0.3286



Powody wyjścia (TBM):


,Liczba
exit_reason,
TRAILING_TP,92
TRAILING_SL,14
SL,1



Powody wyjścia (Simple SL/TP):


,Liczba
exit_reason,
TP,74
SL,30
TIME_EXIT,3


## Optymalizacja

In [13]:
# 6. Optymalizacja Progu Wejścia i Czasu Trzymania Setupu
import itertools
from tqdm.notebook import tqdm

def optimize_threshold(thresholds, max_holding_bars):
    results = []
    
    # Tworzymy siatkę wszystkich kombinacji wejściowych list
    grid = list(itertools.product(thresholds, max_holding_bars))
    
    for th, max_bars in tqdm(grid, desc="Optymalizacja progu"):
        # max_bars definiuje ile dni po sygnale czekamy na wpadnięcie w próg
        ents = generate_odbicie_entries(signals_df, dfs_1d, threshold_pct=th, max_setup_hold_bars=max_bars)
        
        # max_bars definiuje również jak długo trzymamy trade zanim zamkniemy na czas
        trds = moving_triple_barrier_labels(ents, dfs_1d, tp_mult=tpm, sl_mult=slm, tp_trail_mult=ttpm, max_holding_bars=max_bars)
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            results.append({
                'threshold_pct': th,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return
            })
            
    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='avg_return', ascending=False)
    return df_res

'''
thresholds = [9,10,11,12]
max_holding_bars = [15]

print("Uruchamianie optymalizacji progu...")
opt_df = optimize_threshold(thresholds, max_holding_bars)

display(opt_df.head(10))
'''


'\nthresholds = [9,10,11,12]\nmax_holding_bars = [15]\n\nprint("Uruchamianie optymalizacji progu...")\nopt_df = optimize_threshold(thresholds, max_holding_bars)\n\ndisplay(opt_df.head(10))\n'

In [14]:
# 7. Optymalizacja Parametrów TBM (Take Profit / Stop Loss / Max Hold)
import itertools
from tqdm.notebook import tqdm
import pandas as pd

def optimize_tbm(entries_df, market_data_daily, tp_mults, sl_mults, trail_activations, max_holding_bars_list):
    results = []
    
    # Tworzymy siatkę wszystkich kombinacji
    grid = list(itertools.product(tp_mults, sl_mults, trail_activations, max_holding_bars_list))
    
    for tp, sl, trail, max_bars in tqdm(grid, desc="Optymalizacja TBM"):
        trds = moving_triple_barrier_labels(
            entries_df=entries_df, 
            market_data_daily=market_data_daily, 
            tp_mult=tp, 
            sl_mult=sl, 
            tp_trail_mult=trail, 
            max_holding_bars=max_bars,
            early_breakeven=early_bailout,
            time_decay_sl=time_decay_sl,
            active_trailing_sl=active_trail_sl,
            sl_trail_mult=sl_trail_mult,
            max_loss_pct=max_loss_pct
        )
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            avg_hold_bars = trds['hold_bars'].mean()

            results.append({
                'tp_mult': tp,
                'sl_mult': sl,
                'trail_activation': trail,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return,
                'avg_hold_bars': avg_hold_bars,
                'return_per_bar': avg_return / avg_hold_bars if avg_hold_bars > 0 else 0
            })

    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='return_per_bar', ascending=False)
    return df_res


'''
tp_mults = [a/100 for a in range(113, 117, 1)]
sl_mults = [a/100 for a in range(313, 321, 1)]
trail_activations = [a/100 for a in range(1, 3, 1)]
max_holding_bars = [15]

print("Uruchamianie optymalizacji TBM. To może zająć chwilę...")
tbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)

display(tbm_opt_results.head(10))
'''


'\ntp_mults = [a/100 for a in range(113, 117, 1)]\nsl_mults = [a/100 for a in range(313, 321, 1)]\ntrail_activations = [a/100 for a in range(1, 3, 1)]\nmax_holding_bars = [15]\n\nprint("Uruchamianie optymalizacji TBM. To może zająć chwilę...")\ntbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)\n\ndisplay(tbm_opt_results.head(10))\n'